In [ ]:
# RNN으로 다항분류 : 텍스트 생성 (many to many)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical, pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input, Flatten

In [ ]:
text = """
      경마장에 있는 말이 뛰고 있다
      그의 말이 법이다
      가는 말이 고와야 오는 말이 곱다
      """

tok = Tokenizer()    # 단어 단위, 글자 단위로 할 경우 -> Tokenizer(char_level=True)
tok.fit_on_texts([text])
encoded = tok.texts_to_sequences([text])[0]
print(encoded)
print(tok.index_word)    # 단어 사전

vocab_size = len(tok.word_index) + 1
print(f'vocab size : {vocab_size}')

In [ ]:
# train data 만들기
sequences = list()
for line in text.split('\n'):
  enco = tok.texts_to_sequences([line])[0]    # sequences는 list 형태로 입력 받음
  # print(enco)
  for i in range(1, len(enco)):    # 바로 다음 단어를 label로 사용하기 위함
    sequ = enco[:i + 1]
    sequences.append(sequ)

# print(sequences)
print(f"학습에 참여할 샘풀 수 : {len(sequences)}")
print(max(len(i) for i in sequences))    # 제일 긴 단어 6
max_len = max(len(i) for i in sequences)

psequences = pad_sequences(sequences=sequences, maxlen=max_len, padding='pre')
print(psequences)    # 패딩을 하는 이유 : 서로 길이가 다른 입력 시퀀스(문장)들을 하나의 고정된 크기 행렬(텐서)로 묶어 병렬 연산을 쉽게 하기 위함

x = psequences[:, :-1] # features
y = psequences[:, -1] # labels -> OneHot 처리 해줘야함
# print(x)
print(y)

# 레이블 OneHot 처리
y = to_categorical(y, num_classes=vocab_size)
print(y)

In [ ]:
# 모델 생성
model = Sequential()
model.add(Embedding(vocab_size, 32, mask_zero=True))    # mask_zero=True : 앞 쪽에 zero padding을 RNN이 자동으로 무시, 성능/안정성 향상
model.add(LSTM(32, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(x, y, epochs=180, verbose=2, batch_size=32)
print(f"정확도 : {model.evaluate(x, y)[1]:.4f}")

In [ ]:
# 문자열 생성
import numpy as np

def sequences_gen_text(model, tok, current_word, n):
  init_word = current_word
  sentence = ''
  for _ in range(n):
    encoded = tok.texts_to_sequences([current_word])[0]
    encoded = pad_sequences([encoded], maxlen=max_len - 1, padding='pre')
    result = np.argmax(model.predict(encoded, verbose=0), axis=-1)

    # 예측 단어 찾기
    for word, index in tok.word_index.items():
      # print(word, index)
      if index == result:    # 예측한 단어와 인덱스가 동일하면 해당 단어가 예측 단어이므로 break
        break

    current_word = current_word + ' ' + word    # 단어 누적
    sentence = sentence + ' ' + word

  sentence = init_word + sentence
  return sentence  

print(sequences_gen_text(model, tok, '경마', 5))
print(sequences_gen_text(model, tok, '그의', 5))
print(sequences_gen_text(model, tok, '가는', 5))
print(sequences_gen_text(model, tok, '경마장', 5))